# Description

In this notebook, we benchmark PySR algorithm on the Korns benchmarks.

In [ ]:
from config.korns_config import BENCH, PYSR, FEATURE_NAMES
from src.korns_core import RunConfig, load_korns_hdf5, run_benchmark, init_results_csv, append_results_csv_row, SRFitResult

from pysr import PySRRegressor
from dataclasses import dataclass
from typing import Optional, Dict, Any
import numpy as np
import sympy as sp

import warnings
warnings.filterwarnings(
    "ignore",
    message=r"You are using the `\^` operator, but have not set up `constraints` for it\.",
    category=UserWarning,
    module=r"pysr\.sr",
)
warnings.filterwarnings(
    "ignore",
    message=r"Note: it looks like you are running in Jupyter\. The progress bar will be turned off\.",
    category=UserWarning,
    module=r"pysr\.sr",
)


@dataclass
class PySRKornsRegressor:
    name: str = "pysr"
    niterations: int = PYSR.niterations
    populations: int = PYSR.populations
    maxsize: int = PYSR.maxsize
    timeout_in_seconds: Optional[int] = PYSR.timeout_in_seconds

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        model = PySRRegressor(
            niterations=self.niterations,
            populations=self.populations,
            maxsize=self.maxsize,
            unary_operators=list(PYSR.unary_ops),
            binary_operators=list(PYSR.binary_ops),
            elementwise_loss=PYSR.elementwise_loss,
            model_selection=PYSR.model_selection,
            verbosity=PYSR.verbosity,
            progress=PYSR.progress,
            temp_equation_file=PYSR.temp_equation_file,
            delete_tempfiles=PYSR.delete_tempfiles,
            timeout_in_seconds=self.timeout_in_seconds,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        try:
            expr = model.sympy()
        except Exception:
            expr = None
        return SRFitResult(expr=expr, y_pred_test=np.asarray(y_pred, dtype=np.float64), metadata=None)


cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

datasets = load_korns_hdf5(cfg.hdf5_path)

# rows = run_benchmark(
#     datasets=datasets,
#     algorithms=[PySRKornsRegressor()],
#     config=cfg,
#     n_runs=BENCH.n_runs,
#     feature_names=FEATURE_NAMES,
# )

# save_results_csv(rows, PYSR.results_csv_path)

rows = run_benchmark(
    datasets=datasets,
    algorithms=[PySRKornsRegressor()],
    config=cfg,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path=PYSR.results_csv_path,
)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
[PROBLEM] P1
[GT] 24.3*x3 + 1.57
[ALGO] pysr
[RUN START] run_id=0 seed=28732350661000
